In [1]:
import pandas as pd, numpy as np
df2 = pd.read_parquet("/Users/shraddha/Documents/DATA_SCIENCE/Shellscope/shellscope/data/processed/shellscope_companies.parquet")
print(df2.shape)
print(df2.columns.tolist())

addr_cols = [c for c in df2.columns if c.startswith("RegAddress")]
print(addr_cols)
print(df2[addr_cols].isna().mean().round(4))

print(df2["RegAddress.AddressLine1"].sample(15, random_state=1).tolist())
print(df2["RegAddress.AddressLine1"].value_counts().head(15))

(5704711, 23)
['CompanyName', 'CompanyNumber', 'RegAddress.AddressLine1', 'RegAddress.PostTown', 'RegAddress.PostCode', 'CompanyCategory', 'CompanyStatus', 'IncorporationDate', 'Accounts.NextDueDate', 'Accounts.AccountCategory', 'SICCode.SicText_1', 'ConfStmtNextDueDate', 'in_scope', 'company_age_years', 'accounts_overdue', 'confstmt_overdue', 'is_active', 'strike_off_proposal', 'sic_code', 'sic_section', 'postcode_clean', 'postcode_area', 'glasgow']
['RegAddress.AddressLine1', 'RegAddress.PostTown', 'RegAddress.PostCode']
RegAddress.AddressLine1    0.0111
RegAddress.PostTown        0.0166
RegAddress.PostCode        0.0160
dtype: float64
['2 THEATRE SQUARE', '34 BANK STREET', '2 MOUNTSIDE', '17 KING STREET', 'PIKEHELVE ST.', '40 STRATHMORE ROAD', '50 GRASMERE', '5 HAMILTON DRIVE', 'EPIC HOUSE', 'MEADOW VIEW A48', 'BUXUS WINGFIELD ROAD', 'UNIT 3 WOODHAM ROAD', '1 GOLDSMITH CLOSE', 'C/O PROPERTY ACCOUNTS LIMITED', 'C/O. 18 BEEHIVE LANE']
RegAddress.AddressLine1
71-75 SHELTON STREET      

In [2]:
%%time
line1 = (df2["RegAddress.AddressLine1"].astype("string[pyarrow]")
         .str.upper()
         .str.replace(r"^C/O\.?\s*", "", regex=True)
         .str.replace(r"[^\w\s/-]", " ", regex=True)
         .str.replace(r"\s+", " ", regex=True)
         .str.strip()
         .str.replace(r" ST$", " STREET", regex=True)
         .str.replace(r" RD$", " ROAD", regex=True))

df2["address_key"] = line1 + " | " + df2["postcode_clean"].astype("string[pyarrow]")
del line1

df2["companies_at_address"] = df2.groupby("address_key", observed=True)["CompanyNumber"].transform("size")
df2["companies_at_postcode"] = df2.groupby("postcode_clean", observed=True)["CompanyNumber"].transform("size")

CPU times: user 3.71 s, sys: 121 ms, total: 3.83 s
Wall time: 3.86 s


In [3]:
print("missing key:", df2["address_key"].isna().mean().round(4))
print("distinct addresses:", df2["address_key"].nunique())

top = (df2.groupby("address_key", observed=True)
          .agg(n=("CompanyNumber", "size"), town=("RegAddress.PostTown", "first"))
          .sort_values("n", ascending=False)
          .head(15))
print(top)

print(df2[["companies_at_address", "companies_at_postcode"]]
      .describe(percentiles=[.5, .9, .99, .999]).round(1))

print(df2.loc[df2["address_key"].str.startswith("3RD FLOOR", na=False), "address_key"]
         .value_counts().head(5))

missing key: 0.016
distinct addresses: 2632476
                                            n        town
address_key                                              
71-75 SHELTON STREET | WC2H9JQ          87403      LONDON
128 CITY ROAD | EC1V2NX                 43269      LONDON
167-169 GREAT PORTLAND STREET | W1W5PF  25974      LONDON
20 WENLOCK ROAD | N17GU                 20661      LONDON
124 CITY ROAD | EC1V2NX                 18638      LONDON
20-22 WENLOCK ROAD | N17GU              15142      LONDON
3RD FLOOR 86-90 | EC2A4NE               13897      LONDON
27 OLD GLOUCESTER STREET | WC1N3AX      13620      LONDON
66 PAUL STREET | EC2A4NA                11850      LONDON
86-90 PAUL STREET | EC2A4NE              8885      LONDON
61 BRIDGE STREET | HR53DJ                8258     KINGTON
82A JAMES CARTER ROAD | IP287DE          7683  MILDENHALL
50 LOTHIAN ROAD | EH39WJ                 6534   EDINBURGH
1 LYRIC SQUARE | W60NB                   5563      LONDON
124-128 CITY ROAD | EC1V2

In [4]:
bins   = [0, 1, 9, 99, 499, 999, np.inf]
labels = ["1 (alone)", "2-9", "10-99", "100-499", "500-999", "1000+"]
df2["address_band"] = pd.cut(df2["companies_at_address"], bins=bins, labels=labels)

comp = df2["address_band"].value_counts(sort=False)
print(pd.DataFrame({"companies": comp, "share": (comp / comp.sum()).round(4)}))

addr_sizes = df2.groupby("address_key", observed=True).size()
a = pd.cut(addr_sizes, bins=bins, labels=labels).value_counts(sort=False)
print(pd.DataFrame({"addresses": a, "share": (a / a.sum()).round(4)}))

              companies   share
address_band                   
1 (alone)       2103921  0.3748
2-9             1341707  0.2390
10-99            830101  0.1479
100-499          721151  0.1285
500-999          165126  0.0294
1000+            451179  0.0804
           addresses   share
1 (alone)    2103921  0.7992
2-9           493896  0.1876
10-99          30481  0.0116
100-499         3827  0.0015
500-999          247  0.0001
1000+            104  0.0000


In [5]:
o1 = df2.loc[df2["in_scope"] & df2["is_active"], ["address_band", "accounts_overdue"]]
o2 = df2.loc[df2["in_scope"], ["address_band", "strike_off_proposal"]]

r1 = o1.groupby("address_band", observed=True).agg(
        n_active=("accounts_overdue", "size"),
        overdue_rate=("accounts_overdue", "mean"))
r2 = o2.groupby("address_band", observed=True).agg(
        n_in_scope=("strike_off_proposal", "size"),
        strikeoff_rate=("strike_off_proposal", "mean"))

rates = r1.join(r2)
rates["overdue_vs_base"]   = (rates["overdue_rate"]   / o1["accounts_overdue"].mean()).round(2)
rates["strikeoff_vs_base"] = (rates["strikeoff_rate"] / o2["strike_off_proposal"].mean()).round(2)
print("baselines:", round(o1["accounts_overdue"].mean(), 4), round(o2["strike_off_proposal"].mean(), 4))
print(rates.round(4))

baselines: 0.0188 0.0765
              n_active  overdue_rate  n_in_scope  strikeoff_rate  \
address_band                                                       
1 (alone)      1862266        0.0157     2090102          0.0982   
2-9            1211693        0.0200     1329181          0.0763   
10-99           736332        0.0250      811695          0.0547   
100-499         634693        0.0173      704173          0.0498   
500-999         144730        0.0185      158684          0.0577   
1000+           401679        0.0198      433689          0.0632   

              overdue_vs_base  strikeoff_vs_base  
address_band                                      
1 (alone)                0.84               1.28  
2-9                      1.06               1.00  
10-99                    1.33               0.72  
100-499                  0.92               0.65  
500-999                  0.98               0.75  
1000+                    1.05               0.83  


In [6]:
df2["od_active"]   = df2["accounts_overdue"].astype(float).where(df2["in_scope"] & df2["is_active"])
df2["so_in_scope"] = df2["strike_off_proposal"].astype(float).where(df2["in_scope"])
print(df2[["od_active", "so_in_scope"]].mean().round(4))

od_active      0.0188
so_in_scope    0.0765
dtype: float64


In [7]:
big = df2.loc[df2["companies_at_address"] >= 500,
              ["address_key", "RegAddress.PostTown", "od_active", "so_in_scope", "company_age_years"]]

per_addr = (big.groupby("address_key", observed=True)
               .agg(n=("od_active", "size"),
                    town=("RegAddress.PostTown", "first"),
                    overdue_rate=("od_active", "mean"),
                    strikeoff_rate=("so_in_scope", "mean"),
                    median_age=("company_age_years", "median"))
               .sort_values("n", ascending=False))

print(per_addr.head(15).round(3))
print(per_addr[["overdue_rate", "strikeoff_rate", "median_age"]].describe().round(3))
print(per_addr.sort_values("strikeoff_rate", ascending=False).head(10).round(3))

                                            n        town  overdue_rate  \
address_key                                                               
71-75 SHELTON STREET | WC2H9JQ          87403      LONDON         0.018   
128 CITY ROAD | EC1V2NX                 43269      LONDON         0.019   
167-169 GREAT PORTLAND STREET | W1W5PF  25974      LONDON         0.019   
20 WENLOCK ROAD | N17GU                 20661      LONDON         0.023   
124 CITY ROAD | EC1V2NX                 18638      LONDON         0.019   
20-22 WENLOCK ROAD | N17GU              15142      LONDON         0.033   
3RD FLOOR 86-90 | EC2A4NE               13897      LONDON         0.021   
27 OLD GLOUCESTER STREET | WC1N3AX      13620      LONDON         0.025   
66 PAUL STREET | EC2A4NA                11850      LONDON         0.004   
86-90 PAUL STREET | EC2A4NE              8885      LONDON         0.029   
61 BRIDGE STREET | HR53DJ                8258     KINGTON         0.018   
82A JAMES CARTER ROAD | I

In [8]:
df2["age_group"] = pd.cut(df2["company_age_years"], bins=[0, 2, 5, 10, np.inf],
                          labels=["<2y", "2-5y", "5-10y", "10y+"], right=False)

sub = df2.loc[df2["in_scope"], ["age_group", "address_band", "so_in_scope", "od_active"]]

print(sub.groupby("address_band", observed=True)["age_group"]
         .value_counts(normalize=True).unstack().round(3))

print(sub.groupby(["age_group", "address_band"], observed=True)["so_in_scope"]
         .mean().unstack("address_band").round(3))

print(sub.groupby(["age_group", "address_band"], observed=True)["od_active"]
         .mean().unstack("address_band").round(3))

age_group       <2y   2-5y  5-10y   10y+
address_band                            
1 (alone)     0.283  0.197  0.218  0.302
2-9           0.255  0.201  0.231  0.313
10-99         0.170  0.182  0.242  0.406
100-499       0.173  0.192  0.250  0.384
500-999       0.212  0.211  0.254  0.323
1000+         0.439  0.239  0.184  0.138
address_band  1 (alone)    2-9  10-99  100-499  500-999  1000+
age_group                                                     
<2y               0.078  0.059  0.036    0.037    0.040  0.054
2-5y              0.076  0.065  0.045    0.042    0.059  0.065
5-10y             0.176  0.134  0.093    0.074    0.087  0.086
10y+              0.075  0.056  0.044    0.044    0.046  0.060
address_band  1 (alone)    2-9  10-99  100-499  500-999  1000+
age_group                                                     
<2y               0.007  0.009  0.009    0.007    0.006  0.009
2-5y              0.023  0.024  0.026    0.019    0.024  0.031
5-10y             0.018  0.022  0.026    0

In [9]:
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.multitest import multipletests
from scipy import stats

rows = []
for ag in ["<2y", "2-5y", "5-10y", "10y+"]:
    s = sub[(sub["age_group"] == ag) & sub["od_active"].notna()]
    a = s.loc[s["address_band"] == "1000+", "od_active"]
    b = s.loc[s["address_band"] == "1 (alone)", "od_active"]
    z, p = proportions_ztest(count=[a.sum(), b.sum()], nobs=[len(a), len(b)])
    rows.append({"age": ag, "n_1000+": len(a), "rate_1000+": a.mean(),
                 "n_alone": len(b), "rate_alone": b.mean(),
                 "ratio": a.mean() / b.mean(),
                 "diff_pp": 100 * (a.mean() - b.mean()),
                 "z": z, "p": p})

res = pd.DataFrame(rows)
res["p_holm"] = multipletests(res["p"], method="holm")[1]
print(res.round(4))

     age  n_1000+  rate_1000+  n_alone  rate_alone   ratio  diff_pp        z  \
0    <2y   180230      0.0087   544574      0.0073  1.1872   0.1369   5.7818   
1   2-5y    96192      0.0312   380326      0.0230  1.3571   0.8220  14.6803   
2  5-10y    70916      0.0253   372531      0.0178  1.4150   0.7407  13.2312   
3   10y+    54341      0.0293   564835      0.0176  1.6682   1.1728  19.3370   

     p  p_holm  
0  0.0     0.0  
1  0.0     0.0  
2  0.0     0.0  
3  0.0     0.0  


In [10]:
ct = pd.crosstab(sub["address_band"], sub["so_in_scope"])
print(ct)
chi2, p, dof, expected = stats.chi2_contingency(ct)
n = ct.to_numpy().sum()
cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
print(f"chi2={chi2:.1f}  dof={dof}  p={p:.3g}  Cramer's V={cramers_v:.3f}")

so_in_scope       0.0     1.0
address_band                 
1 (alone)     1884810  205292
2-9           1227804  101377
10-99          767286   44409
100-499        669121   35052
500-999        149535    9149
1000+          406277   27412
chi2=28423.8  dof=5  p=0  Cramer's V=0.072


In [11]:
resid = (ct - expected) / np.sqrt(expected)
print(resid.round(1))

so_in_scope    0.0    1.0
address_band             
1 (alone)    -32.7  113.7
2-9            0.2   -0.8
10-99         20.4  -70.9
100-499       23.3  -81.0
500-999        7.8  -27.1
1000+          9.1  -31.6


In [12]:
import numpy as np, pandas as pd
from scipy import stats
from statsmodels.stats.proportion import proportion_confint
rng = np.random.default_rng(42)

mass = df2.loc[(df2["address_band"] == "1000+") & df2["od_active"].notna(),
               ["address_key", "age_group", "od_active"]]

def cluster_boot(d, B=10_000):
    g = d.groupby("address_key", observed=True)["od_active"].agg(overdue="sum", n="count")
    idx = rng.integers(0, len(g), size=(B, len(g)))
    boot = g["overdue"].to_numpy()[idx].sum(axis=1) / g["n"].to_numpy()[idx].sum(axis=1)
    return g, boot

for label, d in [("all ages", mass), ("10y+", mass[mass["age_group"] == "10y+"])]:
    g, boot = cluster_boot(d)
    k, n = g["overdue"].sum(), g["n"].sum()
    lo, hi = proportion_confint(k, n, method="wilson")
    clo, chi = np.percentile(boot, [2.5, 97.5])
    print(f"{label}: companies={n}, addresses={len(g)}, rate={k/n:.4f} | "
          f"naive CI {lo:.4f}-{hi:.4f} | cluster CI {clo:.4f}-{chi:.4f} | "
          f"cluster CI is {(chi-clo)/(hi-lo):.1f}x wider")

all ages: companies=401679, addresses=100, rate=0.0198 | naive CI 0.0194-0.0202 | cluster CI 0.0174-0.0231 | cluster CI is 6.6x wider
10y+: companies=54341, addresses=100, rate=0.0293 | naive CI 0.0279-0.0307 | cluster CI 0.0205-0.0419 | cluster CI is 7.6x wider


In [13]:
alone10 = sub.loc[(sub["age_group"] == "10y+") & (sub["address_band"] == "1 (alone)"),
                  "od_active"].dropna()
pa, na = alone10.mean(), len(alone10)
alone_boot = rng.binomial(na, pa, size=10_000) / na

g10, boot10 = cluster_boot(mass[mass["age_group"] == "10y+"])
ratio_boot = boot10 / alone_boot
print("ratio 2.5% / 50% / 97.5%:", np.percentile(ratio_boot, [2.5, 50, 97.5]).round(2))

ratio 2.5% / 50% / 97.5%: [1.16 1.63 2.4 ]


In [14]:
x = df2.loc[df2["in_scope"] & (df2["address_band"] == "1000+"), "company_age_years"].dropna()
y = df2.loc[df2["in_scope"] & (df2["address_band"] == "1 (alone)"), "company_age_years"].dropna()
u, p = stats.mannwhitneyu(x, y, alternative="two-sided")
print(f"n={len(x)} vs {len(y)} | median age {x.median():.2f} vs {y.median():.2f} | "
      f"p={p:.2e} | P(mass-address company is older) = {u/(len(x)*len(y)):.3f}")

n=433689 vs 2090102 | median age 2.57 vs 5.43 | p=0.00e+00 | P(mass-address company is older) = 0.368


In [15]:
df2["region"] = np.where(df2["glasgow"], "Glasgow (G)", "Rest of UK")
print(df2["region"].value_counts())
print(pd.crosstab(df2["region"], df2["address_band"], normalize="index").round(3))

region
Rest of UK     5621030
Glasgow (G)      83681
Name: count, dtype: int64
address_band  1 (alone)    2-9  10-99  100-499  500-999  1000+
region                                                        
Glasgow (G)       0.372  0.259  0.166    0.113    0.045  0.045
Rest of UK        0.375  0.239  0.148    0.129    0.029  0.081


In [16]:
gla = (df2[df2["glasgow"] & (df2["companies_at_address"] >= 100)]
         .groupby("address_key", observed=True)
         .agg(n=("CompanyNumber", "size"),
              overdue_rate=("od_active", "mean"),
              strikeoff_rate=("so_in_scope", "mean"),
              median_age=("company_age_years", "median"))
         .sort_values("n", ascending=False))
print("Glasgow addresses with 100+ companies:", len(gla))
print(gla.head(10).round(3))
print(gla[["overdue_rate", "strikeoff_rate"]].describe().round(3))

Glasgow addresses with 100+ companies: 59
                                      n  overdue_rate  strikeoff_rate  \
address_key                                                             
272 BATH STREET | G24JR            1614         0.042           0.381   
48 WEST GEORGE STREET | G21BP      1099         0.018           0.068   
CLYDE OFFICES 2ND FLOOR | G21BP    1046         0.016           0.054   
6TH FLOOR GORDON CHAMBERS | G13NQ   916         0.021           0.058   
1 CAMBUSLANG COURT | G328FH         898         0.006           0.020   
9 ROYAL CRESCENT | G37SP            817         0.011           0.086   
HORIZON CA | G37LH                  594         0.021           0.189   
OFFICE 1 | G413JA                   573           NaN             NaN   
2ND FLOOR | G24BG                   499         0.013           0.062   
14 NEWTON PLACE | G37PY             431         0.010           0.049   

                                   median_age  
address_key                      

In [18]:
## '''Part 2 conclusions
1. **Concentration:** 351 addresses (0.01%) hold 11% of all UK companies; the largest holds 87,403.
2. **No simple "big address = risk" pattern:** strike-off is highest for companies alone at their address (1.28× baseline).
   Address band and strike-off are linked but weakly (χ² = 28,424, df = 5, p < 0.001, Cramér's V = 0.07).
3. **Overdue accounts, age-adjusted:** within every age group, companies at 1000+ addresses are more often overdue
   (1.19× for <2y up to 1.67× for 10y+; Holm-adjusted p < 0.001). This was hidden in the raw totals because mass-address
   companies are much younger (median 2.6 vs 5.4 years; Mann-Whitney p < 0.001) → Simpson's paradox.
4. **Honest uncertainty:** companies cluster by agent. A cluster bootstrap over addresses gives 1.63 (95% CI 1.16–2.40),
   about 7× wider than the naive interval. The effect holds but its size is uncertain.
5. **Heterogeneity:** most big addresses look ordinary; a few have extreme strike-off rates (>70%).
   The signal is "unusual address", not "big address".
6. **Glasgow:** … (Step 2.7)

**Risk indicator, not proof.** A shared address is a legal, common service. Even where rates are higher, about 97%
of flagged companies are not overdue, and because genuine shell companies are rare, any address-based flag would
mostly point at legitimate businesses (base-rate effect). These indicators can prioritise further checks; they are
not evidence of wrongdoing. No individual companies or people are named in this analysis.'''

SyntaxError: invalid character '×' (U+00D7) (3597265637.py, line 3)

In [19]:
from pathlib import Path
DATA = Path("/Users/shraddha/Documents/DATA_SCIENCE/Shellscope/shellscope/data/processed")
df2.to_parquet(DATA / "shellscope_part2.parquet", index=False)
print(pd.read_parquet(DATA / "shellscope_part2.parquet").shape)

(5704711, 31)


In [ ]:
ftyz